# Import Mount Isa from OpenStreetMap

This demo keeps two local input options. The default uses OSMnx to pre-download and cache OSM responses in python/data/osm/osmnx_cache, then AequilibraE imports from that cache. After the first download, reuse or share that directory for a fast repeatable demo. The alternate option imports a prepared .osm.pbf file through pyrosm.

For the OSMnx cache option, no third-party extract service is used: OSMnx fetches the compact Mount Isa study area from OpenStreetMap services. The saved cache contains OSM HTTP response JSON, not a .osm.pbf file. Keep the cache directory and use the same study area, OSMnx version, and request settings when replaying it.

The live area is centered on the top-ranked Mount Isa town result with a 5 km radius, avoiding the much larger administrative boundaries returned by a full place-name search. OSM data is community maintained; retain attribution and review coverage before analysis.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import pandas as pd
from aequilibrae import Project

# A fresh folder lets the notebook be run more than once without replacing an existing project.
project_path = Path(gettempdir()) / f"aequilibrae_mount_isa_{uuid4().hex[:8]}"
print(f"AequilibraE project folder: {project_path}")


AequilibraE project folder: /tmp/aequilibrae_mount_isa_8302a5a2


In [2]:
import logging

from aequilibrae.utils import (
    basic_config,
    ensure_spatialite_binaries,
    set_known_spatialite_folder,
)

logger = logging.getLogger("aequilibrae")
basic_config(logging.DEBUG)

<AequilibraETQDMStreamHandler stdout (NOTSET)>

In [3]:
data_dir = Path.cwd() / "data"
assert data_dir.is_dir()

## Download and crop the Queensland PBF

This downloads the current Geofabrik Queensland extract, then uses Pyrosm to write a Mount Isa PBF crop. The extent matches the coordinates in the osmium example below; Pyrosm expects `[west, south, east, north]`, so the two latitude values are ordered from south to north. The source and crop are kept in `python/data/osm` (or `data/osm` when the notebook is launched from `python`).


In [4]:
from urllib.request import urlretrieve

from pyrosm import OSM

osm_data_dir = data_dir / "osm"

geofabrik_url = "https://download.geofabrik.de/australia-oceania/australia/queensland-latest.osm.pbf"
source_pbf_path = osm_data_dir / "queensland-latest.osm.pbf"
pbf_path = osm_data_dir / "mount-isa.osm.pbf"

bbox = [139.45510, -20.79793, 139.57898, -20.60647]  # west, south, east, north
if not pbf_path.is_file():
    print(f"Downloading {geofabrik_url}")
    urlretrieve(geofabrik_url, source_pbf_path)
    print(f"Cropping Queensland PBF to {bbox}")
    OSM(source_pbf_path, bounding_box=bbox).to_pbf(
        output_path=str(pbf_path), repack=True
    )

print(f"Mount Isa PBF: {pbf_path}")

Mount Isa PBF: /home/jake/OuterLoop/atrf2026/workshop/data/osm/mount-isa.osm.pbf


## Import the cached data into AequilibraE
  With osmnx_cache, the AequilibraE importer repeats the same polygon query and OSMnx serves it from osmnx_cache. Once the cache has been prepared, this can run without reaching Nominatim or Overpass. With pbf, the importer reads the local file through pyrosm.


In [5]:
project = Project.create(project_path)

2026-11-18 16:29:53,937;INFO ; Created project on /tmp/aequilibrae_mount_isa_8302a5a2


In [6]:
project.tables.importer.osm(
    pbf_path=pbf_path,
    modes=("car", "transit", "bicycle", "walk"),
    simplify=True,
)

2026-11-18 16:29:54,090;INFO ; Acquiring network from source 'osm-pbf' (modes=('car', 'transit', 'bicycle', 'walk'))
2026-11-18 16:29:54,826;INFO ; Mode filter kept 27233 / 27533 links
2026-11-18 16:29:54,973;INFO ; Acquired 25832 nodes and 27233 links
2026-11-18 16:29:54,974;INFO ; Simplifying with 'osmnx'
2026-11-18 16:30:02,711;INFO ; After simplification: 2528 nodes, 3798 links
2026-11-18 16:30:02,712;INFO ; Saving network to the project database
2026-11-18 16:30:02,717;DEBUG ; Temporarily removing trigger aequilibrae_cannibalize_node_abort_when_centroid
2026-11-18 16:30:02,718;DEBUG ; Temporarily removing trigger aequilibrae_default_period_delete
2026-11-18 16:30:02,719;DEBUG ; Temporarily removing trigger aequilibrae_default_period_update
2026-11-18 16:30:02,720;DEBUG ; Temporarily removing trigger aequilibrae_deleted_link
2026-11-18 16:30:02,720;DEBUG ; Temporarily removing trigger aequilibrae_dont_delete_node
2026-11-18 16:30:02,721;DEBUG ; Temporarily removing trigger aequilib

## Inspect the imported network

The project keeps its network in the AequilibraE project database. Inspect the imported link and node tables and check the modes represented in the network.

In [7]:
links = project.tables.links.data
nodes = project.tables.nodes.data
print(f"Links: {len(links):,}\nNodes: {len(nodes):,}")
links.head()

Links: 3,798
Nodes: 2,528


,link_id,a_node,b_node,direction,distance,modes,link_type,name,fixed_cost_ab,fixed_cost_ba,speed_ab,speed_ba,travel_time_ab,travel_time_ba,capacity_ab,capacity_ba,other_attributes,geometry
0,1,100000,100001,0,255.113000,bctw,residential,George Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27210446"",""source_ids"":""{\""schem...","LINESTRING (139.4999 -20.722, 139.50227 -20.72..."
1,2,100000,100002,0,54.927000,bctw,residential,Mullan Street,None,None,NaN,NaN,None,None,None,None,"{""lanes_ab"":1.0,""lanes_ba"":1.0,""source_id"":""26...","LINESTRING (139.4999 -20.722, 139.49996 -20.72..."
2,3,100000,100003,0,89.054000,bctw,residential,George Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27210446"",""source_ids"":""{\""schem...","LINESTRING (139.4999 -20.722, 139.49937 -20.72..."
3,4,100001,100004,0,112.880933,bctw,residential,George Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27210446"",""source_ids"":""{\""schem...","LINESTRING (139.50227 -20.72142, 139.50326 -20..."
4,5,100001,100005,0,109.294000,bctw,residential,King Street,None,None,NaN,NaN,None,None,None,None,"{""source_id"":""27247376"",""source_ids"":""{\""schem...","LINESTRING (139.50227 -20.72142, 139.50209 -20..."


In [8]:
links.to_parquet("links.parquet")

In [9]:
zones_file = data_dir / "Mount Isa SA1.parquet"
employment_file = data_dir / "Mount Isa SA1 employment.csv"

In [10]:
zones = gpd.read_parquet(zones_file)[["SA1_CODE_2021", "geometry"]].explode()
zones["SA1_CODE_2021"] = zones["SA1_CODE_2021"].astype(int)
zones = zones.rename(columns={"SA1_CODE_2021": "sa1_code_2021"}).set_index(
    "sa1_code_2021"
)
print(f"Zones: {len(zones):,}")
zones.head()

Zones: 52


,geometry
sa1_code_2021,
31502140501,"POLYGON ((139.49615 -20.7372, 139.49527 -20.73..."
31502140502,"POLYGON ((139.48816 -20.70872, 139.48813 -20.7..."
31502140503,"POLYGON ((139.49131 -20.73395, 139.49142 -20.7..."
31502140504,"POLYGON ((139.4904 -20.72597, 139.49034 -20.72..."
31502140505,"POLYGON ((139.49807 -20.7245, 139.49799 -20.72..."


In [11]:
employment = (
    pd.read_csv(employment_file, skiprows=9)  # Skip ABS header
    .tail(-1)  # Drop the first row which is blank for the two row header
    .head(-5)  # Drop the last 5 rows which contain data source info
    .loc[:, :"Total"]  # Drop columns after "Total", just whitespace
    .rename(
        columns={"1-digit level INDP Industry of Employment": "SA1_CODE_2021"}
    )  # Pandas reads this column name incorrectly due to the multi-row header
    .astype(int)
)

column_names = {
    column: column.lower().replace(" ", "_").replace(",", "")
    for column in employment.columns
} | {"SA1_CODE_2021": "sa1_code_2021"}

employment = employment.rename(columns=column_names).set_index("sa1_code_2021")

print(f"Employment: {len(employment):,}")
employment.head()

Employment: 63


,agriculture_forestry_and_fishing,mining,manufacturing,electricity_gas_water_and_waste_services,construction,wholesale_trade,retail_trade,accommodation_and_food_services,transport_postal_and_warehousing,information_media_and_telecommunications,...,administrative_and_support_services,public_administration_and_safety,education_and_training,health_care_and_social_assistance,arts_and_recreation_services,other_services,inadequately_described,not_stated,not_applicable,total
sa1_code_2021,,,,,,,,,,,,,,,,,,,,,
31502140601,20,22,6,5,13,4,10,7,25,0,...,3,15,7,28,4,12,0,5,175,361
31502140602,4,36,5,3,7,0,9,10,27,0,...,3,17,14,18,0,0,0,4,145,311
31502140603,10,35,4,0,17,0,21,23,31,0,...,10,27,9,16,0,14,0,12,295,519
31502140604,4,25,10,0,25,4,12,3,18,0,...,3,17,23,5,0,5,0,3,153,326
31502140605,5,0,0,0,9,0,3,7,3,0,...,0,6,10,4,0,3,0,0,107,152


In [12]:
zones = zones.join(employment, how="left")

Our dataframe has more columns than our database! Before we store our new zoning system we should create the columns we'd like to keep. This can be done through plan SQLite queries, or through the field editor.

In [13]:
existing_columns = set(project.tables.zones.columns)
columns = set(zones.columns)

missing_columns = columns - existing_columns
print("Missing columns:", missing_columns)

Missing columns: {'health_care_and_social_assistance', 'professional_scientific_and_technical_services', 'electricity_gas_water_and_waste_services', 'arts_and_recreation_services', 'construction', 'administrative_and_support_services', 'public_administration_and_safety', 'retail_trade', 'total', 'mining', 'agriculture_forestry_and_fishing', 'financial_and_insurance_services', 'rental_hiring_and_real_estate_services', 'inadequately_described', 'not_stated', 'wholesale_trade', 'information_media_and_telecommunications', 'manufacturing', 'transport_postal_and_warehousing', 'education_and_training', 'not_applicable', 'other_services', 'accommodation_and_food_services'}


In [14]:
import sqlite3

try:
    with project.db_connection as conn:
        conn.execute('ALTER TABLE zones ADD COLUMN "SA1_CODE_2021" INTEGER')
except sqlite3.OperationalError as e:
    print("Error adding column:", e)

In [15]:
fields = project.tables.zones.fields
existing_fields = fields.all_fields()
print("Existing zone fields:", existing_fields)

for proper_name, column in column_names.items():
    if column not in existing_fields:
        fields.add(column, description=f"ABS: {proper_name}", data_type="integer")
        logger.info("Added field: %s", column)

fields.save()

Existing zone fields: ['area', 'name', 'zone_id', 'geometry', 'sa1_code_2021']
2026-11-18 16:30:03,485;INFO ; Added field: agriculture_forestry_and_fishing
2026-11-18 16:30:03,499;INFO ; Added field: mining
2026-11-18 16:30:03,513;INFO ; Added field: manufacturing
2026-11-18 16:30:03,526;INFO ; Added field: electricity_gas_water_and_waste_services
2026-11-18 16:30:03,540;INFO ; Added field: construction
2026-11-18 16:30:03,554;INFO ; Added field: wholesale_trade
2026-11-18 16:30:03,569;INFO ; Added field: retail_trade
2026-11-18 16:30:03,584;INFO ; Added field: accommodation_and_food_services
2026-11-18 16:30:03,599;INFO ; Added field: transport_postal_and_warehousing
2026-11-18 16:30:03,613;INFO ; Added field: information_media_and_telecommunications
2026-11-18 16:30:03,626;INFO ; Added field: financial_and_insurance_services
2026-11-18 16:30:03,639;INFO ; Added field: rental_hiring_and_real_estate_services
2026-11-18 16:30:03,651;INFO ; Added field: professional_scientific_and_techni

We'll table the zones dataframe that we've constructed and insert it into the database. Because we haven't supplied or set the ``zone_ids`` column ourselves, AequilibraE will infer that from the insertion order. If you need control over the IDs, say for instance, to match some existing IDs, then you should set that manually.

In [16]:
zones["zone_id"] = project.tables.zones.insert_from(zones.reset_index())

Let's now add centroids to each of our zones, and connect them to the ``b`` mode  links with centroid connectors.

In [17]:
project.tables.zones.add_centroids()
project.tables.zones.connect_mode("b")

Adding centroids                                  :   0%|          | 0/52 [00:00<?, ?it/s]

2026-11-18 16:30:04,161;INFO ; 52 new centroids added to the network


Connecting zones                                  :   0%|          | 0/52 [00:00<?, ?it/s]

2026-11-18 16:30:04,308;INFO ; 1 new centroid connectors for mode b added for centroid 1
2026-11-18 16:30:04,347;INFO ; 1 new centroid connectors for mode b added for centroid 2
2026-11-18 16:30:04,385;INFO ; 1 new centroid connectors for mode b added for centroid 3
2026-11-18 16:30:04,424;INFO ; 1 new centroid connectors for mode b added for centroid 4
2026-11-18 16:30:04,461;INFO ; 1 new centroid connectors for mode b added for centroid 5
2026-11-18 16:30:04,499;INFO ; 1 new centroid connectors for mode b added for centroid 6
2026-11-18 16:30:04,536;INFO ; 1 new centroid connectors for mode b added for centroid 7
2026-11-18 16:30:04,573;INFO ; 1 new centroid connectors for mode b added for centroid 8
2026-11-18 16:30:04,611;INFO ; 1 new centroid connectors for mode b added for centroid 9
2026-11-18 16:30:04,648;INFO ; 1 new centroid connectors for mode b added for centroid 10
2026-11-18 16:30:04,686;INFO ; 1 new centroid connectors for mode b added for centroid 11
2026-11-18 16:30:04

In [18]:
import geopandas as gpd
import numpy as np
from aequilibrae.route_choice import RouteChoice
from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap
from matplotlib import colormaps
from shapely.geometry import box


To operate on the road network, we should "build" it. This involves loading all appropriate data from the database include links, nodes, centroids, turn restrictons, U-turn configurations etc for a set of modes. In this case we're just interested in ``b``.

In [19]:
network = project.road.build_network(modes=["b"])

With our network loaded, we can edit and update the network dataframe before building a graph. This allows us to compute arbitary columns at runtime, and have them work just as any other column would.

Here we've picked a pretty arbitary utility function.

In [75]:
network.links["utility"] = 0.01 / 1000 * network.links["distance"]

graph = network.build_graph(
    mode="b",
    cost="distance",
    compact=True,
    remove_dead_ends=True,
    block_centroid_flows=False,
)

In [76]:
rc = RouteChoice(graph)
rc.set_choice_set_generation(algorithm="bfsle", max_routes=5, max_depth=5)

In [77]:
rc.prepare([(15, 49)])

In [78]:
rc.execute()

0/1 ODs processed                                 :   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-18 16:43:18,670;DEBUG ; Route choice: 14, 48


In [79]:
routes = rc.get_results()

In [80]:
routes

,origin id,destination id,cost,mask,path overlap,probability,route set,route directions
0,15,49,10267.403796,True,0.261603,1.952144e-02,"[3813, 3786, 3783, 1412, 1405, 1400, 1396, 139...","[1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1..."
1,15,49,13796.688084,True,0.851607,0.000000e+00,"[3813, 3786, 3783, 1412, 1405, 1404, 1410, 141...","[1, -1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, 1, ..."
2,15,49,10285.693303,True,0.283818,2.414778e-10,"[3813, 3786, 3783, 1412, 1405, 1400, 1396, 139...","[1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1..."
3,15,49,10282.015974,True,0.251192,8.450578e-09,"[3813, 3786, 3783, 3784, 1413, 1405, 1400, 139...","[1, -1, -1, 1, -1, -1, -1, -1, -1, -1, -1, -1,..."
4,15,49,10263.420243,True,0.244644,9.804786e-01,"[3813, 3786, 3783, 1412, 1405, 1400, 1396, 139...","[1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1..."


In [81]:
links = project.tables.links.data
nodes = project.tables.nodes.data

In [82]:
from functools import reduce

In [83]:
route_links_ids = reduce(np.union1d, routes["route set"])
route_links = links[links["link_id"].isin(route_links_ids)]

separate_routes_ids = list(routes["route set"])
separate_routes_ids

[array([3813, 3786, 3783, 1412, 1405, 1400, 1396, 1392, 1390, 1380, 1374,
        1357, 1354, 1350, 1345, 1342, 1259, 1258, 1260,  962,  957,  122,
         121,  971,  999, 1004,  781,  779, 1002,  791, 2388, 1684,  728,
         326,  693,  678,  679,  694,  683,  682, 1679, 1677, 1671, 1672,
         631,  629,  630,  620,  616,  612,  610,  606,  602,  597,  362,
         359, 3847], dtype=uint64),
 array([3813, 3786, 3783, 1412, 1405, 1404, 1410, 1419, 1421, 1425, 1269,
        1266, 1264, 1267, 1271, 1313, 1290, 1285, 1284, 1288, 1292, 1307,
        1303, 1300, 1296, 1294, 1295, 1298, 1316, 1256, 1250, 1245, 1244,
         276,  269,  258,  259,  270, 1133, 1132, 1139, 1153, 1199, 1192,
        1179, 1169, 1165, 1120,  913,  911,  910, 1115, 1110,  845,  840,
         839,  863, 2205,  718,  716, 2392, 2390, 2244,  614,  613,  610,
         606,  602,  597,  362,  359, 3847], dtype=uint64),
 array([3813, 3786, 3783, 1412, 1405, 1400, 1396, 1392, 1390, 1380, 1374,
        1357, 13

In [84]:
route_links_ids

array([ 121,  122,  258,  259,  269,  270,  276,  326,  359,  362,  597,
        602,  606,  610,  612,  613,  614,  616,  620,  629,  630,  631,
        678,  679,  682,  683,  693,  694,  716,  718,  728,  779,  781,
        791,  839,  840,  845,  863,  910,  911,  913,  957,  961,  962,
        969,  971,  972,  978,  983,  999, 1002, 1004, 1110, 1115, 1120,
       1132, 1133, 1139, 1153, 1165, 1169, 1179, 1192, 1199, 1244, 1245,
       1250, 1256, 1258, 1259, 1260, 1264, 1266, 1267, 1269, 1271, 1284,
       1285, 1288, 1290, 1292, 1294, 1295, 1296, 1298, 1300, 1303, 1307,
       1313, 1316, 1342, 1345, 1350, 1354, 1357, 1374, 1380, 1390, 1392,
       1396, 1400, 1404, 1405, 1410, 1412, 1413, 1419, 1421, 1425, 1671,
       1672, 1677, 1679, 1684, 2205, 2244, 2388, 2390, 2392, 3783, 3784,
       3786, 3813, 3847], dtype=uint64)

In [85]:
hover_links = links[
    [
        "link_id",
        "link_type",
        "direction",
        "geometry",
    ]
].rename(
    columns={
        "link_id": "Link ID",
        "link_type": "Link type",
        "direction": "Direction",
    }
)

link_layer = PathLayer.from_geopandas(
    hover_links,
    width_units="pixels",
    width_min_pixels=1,
    width_max_pixels=8,
    auto_highlight=True,
)
rc_layer = PathLayer.from_geopandas(
    route_links[["geometry"]],
    width_units="pixels",
    width_min_pixels=4,
    width_max_pixels=128,
    get_color=[0, 0, 255],
    pickable=False,
)

nodes_layer = ScatterplotLayer.from_geopandas(
    nodes[nodes["is_centroid"] != 1][
        [
            "node_id",
            "geometry",
        ]
    ],
    get_fill_color=[255, 0, 0],
    get_radius=10,
)
centroid_layer = ScatterplotLayer.from_geopandas(
    nodes[nodes["is_centroid"] == 1][
        [
            "node_id",
            "geometry",
        ]
    ],
    get_fill_color=[0, 255, 0],
    get_radius=20,
)

# 139.45510,-20.60647,139.57898,-20.79793
bbox = PolygonLayer.from_geopandas(
    gpd.GeoDataFrame(
        geometry=[box(139.45510, -20.79793, 139.57898, -20.60647)], crs="EPSG:4326"
    ),
    get_fill_color=[100, 100, 100, 128],
)

traffic_map = Map(
    layers=[bbox, link_layer, rc_layer, nodes_layer, centroid_layer],
    basemap=None,
    height="1200px",
    show_side_panel=False,
    show_tooltip=True,
)
traffic_map

In [ ]:
def make_pretty_route_choice(
    source,
    dest,
    rc: RouteChoice,
    routes,
    project: Project,
    separate_routes: bool = True,
):
    routes = routes[routes["mask"] == 1]
    routes = routes.sort_values(by="cost", ascending=False)  # sort descending

    route_links_ids = reduce(np.union1d, routes["route set"])
    route_links = links[links["link_id"].isin(route_links_ids)]
    separate_routes_ids = list(routes["route set"])
    separate_routes_links = [
        links[links["link_id"].isin(ids)] for ids in separate_routes_ids
    ]

    hover_links = links[
        [
            "link_id",
            "link_type",
            "direction",
            "geometry",
        ]
    ].rename(
        columns={
            "link_id": "Link ID",
            "link_type": "Link type",
            "direction": "Direction",
        }
    )

    link_layer = PathLayer.from_geopandas(
        hover_links,
        width_units="pixels",
        width_min_pixels=1,
        width_max_pixels=8,
        auto_highlight=True,
    )

    n_routes = len(separate_routes_links)
    normalized = np.linspace(0, 1, n_routes)
    turbo_colors = apply_continuous_cmap(normalized, colormaps["turbo"])
    # turbo_colors is an array of shape (n_routes, 4), values 0-255
    MIN_WIDTH, MAX_WIDTH = 4, 24  # pixels
    widths = np.linspace(MAX_WIDTH, MIN_WIDTH, n_routes)

    rc_layers: list[PathLayer] = []
    if separate_routes:
        for i, (single_route_links, color, width) in enumerate(
            zip(separate_routes_links, turbo_colors, widths)
        ):
            rc_layer = PathLayer.from_geopandas(
                single_route_links[["geometry"]],
                width_units="pixels",
                get_width=float(width),
                width_min_pixels=1,
                width_max_pixels=MAX_WIDTH,
                get_color=color.tolist(),
                pickable=False,
            )
            rc_layers.append(rc_layer)
    else:
        rc_layer = PathLayer.from_geopandas(
            route_links[["geometry"]],
            width_units="pixels",
            width_min_pixels=4,
            # width_max_pixels=128,
            get_color=[0, 0, 255],
            pickable=False,
        )
        rc_layers.append(rc_layer)

    nodes_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] != 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[255, 0, 0],
        get_radius=10,
    )
    centroid_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] == 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[0, 255, 0],
        get_radius=20,
    )

    # 139.45510,-20.60647,139.57898,-20.79793
    bbox = PolygonLayer.from_geopandas(
        gpd.GeoDataFrame(
            geometry=[box(139.45510, -20.79793, 139.57898, -20.60647)], crs="EPSG:4326"
        ),
        get_fill_color=[100, 100, 100, 128],
    )

    traffic_map = Map(
        layers=[bbox, link_layer, *rc_layers, nodes_layer, centroid_layer],
        basemap=None,
        height="1200px",
        show_side_panel=False,
        show_tooltip=True,
    )
    return traffic_map

In [ ]:
low_cutoff_rc = RouteChoice(graph)
my_cutoff = 1.05
low_cutoff_rc.set_choice_set_generation(
    algorithm="LP",
    max_routes=15,
    max_depth=10,
    disutility_cutoff_coefficient=my_cutoff,
    disutility_cutoff_constant=0,
)

low_cutoff_rc.prepare([(14, 50)])
low_cutoff_rc.execute(perform_assignment=True)
routes = low_cutoff_rc.get_results()

routes = routes.sort_values(by="probability", ascending=False)
routes

0/1 ODs processed                                 :   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-18 16:44:49,670;DEBUG ; Route choice: 13, 49


,origin id,destination id,cost,mask,path overlap,probability,route set,route directions
7,14,50,7145.571871,True,0.201210,9.963172e-01,"[3812, 1984, 1837, 1831, 1834, 1826, 1825, 960...","[1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, 1, 1,..."
6,14,50,7151.432978,True,0.260752,3.677290e-03,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, 1, -1, -1, -1, ..."
5,14,50,7157.880014,True,0.201438,4.503280e-06,"[3812, 1984, 1837, 1831, 1834, 1826, 1825, 960...","[1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, 1, 1,..."
1,14,50,7159.725018,True,0.261608,9.241972e-07,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, 1, 1,..."
4,14,50,7162.698014,True,0.206651,3.734177e-08,"[3812, 1984, 1837, 1831, 1834, 1826, 1825, 960...","[1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, 1, 1,..."
3,14,50,7170.489651,True,0.222412,1.660548e-11,"[3812, 1984, 1837, 1831, 1834, 1826, 1825, 960...","[1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, 1, 1,..."
2,14,50,7511.299332,False,0.000000,0.000000e+00,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, 1, -1..."
0,14,50,7511.912766,False,0.000000,0.000000e+00,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -..."


In [93]:
make_pretty_route_choice(14, 50, low_cutoff_rc, routes, project, separate_routes=True)